In [3]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import QasmSimulator
from collections import defaultdict
import networkx as nx
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit.circuit import ParameterVector, Parameter

# from qiskit_braket_provider import BraketProvider


In [4]:
service = QiskitRuntimeService()
backends = {}
# backends["ibm_fez"] = service.backend("ibm_fez", use_fractional_gates=True)
# backends["ibm_marrakesh"] = service.backend("ibm_marrakesh")
# backends["ibm_torino"] = service.backend("ibm_torino")
# backends["ibm_brisbane"] = service.backend("ibm_brisbane")
# backends["ibm_sherbrooke"] = service.backend("ibm_sherbrooke")
# backends["ibm_kyiv"] = service.backend("ibm_kyiv")
# backends["mps"] = service.backend("simulator_mps")
backends["qasm_simulator"] = QasmSimulator()



# QAOA Circuit 

## LABS 

In [5]:
import math
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector


# ----------------------------------------------------------------------
# Helper 1: build LABS Hamiltonian terms for arbitrary N
# ----------------------------------------------------------------------
def build_labs_terms(N):
    """
    Build the LABS Hamiltonian terms for sequence length N, using
    the form from Shaydulin et al. (Eq. 4), but in 0-based indexing.

    Returns:
        terms: list of (qubits, coeff)
            - qubits: tuple of logical qubit indices, e.g. (0, 2) or (0, 1, 2, 3)
            - coeff: float coefficient in front of Z⊗...⊗Z term
    """
    terms = []

    # 4-local terms: 2 * Z_i Z_{i+t} Z_{i+k} Z_{i+k+t}
    # paper: i = 1..N-3 (1-based), t = 1..floor((N-i-1)/2), k = t+1..N-i-t
    # here we convert to 0-based indices
    for i in range(1, N - 2 + 1):  # i in [1, N-2] (1-based)
        max_t = (N - i - 1) // 2
        for t in range(1, max_t + 1):
            for k in range(t + 1, N - i - t + 1):
                j0 = i - 1
                j1 = i + t - 1
                j2 = i + k - 1
                j3 = i + k + t - 1
                qubits = tuple(sorted([j0, j1, j2, j3]))
                terms.append((qubits, 2.0))  # coefficient 2 in Eq. (4)

    # 2-local terms: Z_i Z_{i+2k}
    # paper: i = 1..N-2, k = 1..floor((N-i)/2)
    for i in range(1, N - 1 + 1):  # i in [1, N-1]
        max_k = (N - i) // 2
        for k in range(1, max_k + 1):
            j0 = i - 1
            j1 = i + 2 * k - 1
            qubits = tuple(sorted([j0, j1]))
            terms.append((qubits, 1.0))  # coefficient 1 in Eq. (4)

    # (Constant offset in Esidelobe is dropped → only a global phase)
    # Deduplicate any accidental duplicates (shouldn't really happen, but just in case)
    unique_terms = {}
    for qubits, coeff in terms:
        unique_terms[qubits] = unique_terms.get(qubits, 0.0) + coeff

    return [(q, c) for q, c in unique_terms.items() if abs(c) > 1e-12]


# ----------------------------------------------------------------------
# Helper 2: apply exp(-i * gamma * coeff * Z⊗...⊗Z) for a given term
# ----------------------------------------------------------------------
def _apply_parity_phase(qc: QuantumCircuit, physical_qubits, gamma: float, coeff: float):
    """
    Apply U = exp(-i * gamma * coeff * Z⊗...⊗Z) on the given *physical* qubits
    using a CNOT ladder into the last qubit, then RZ, then uncompute.

    Args:
        qc: QuantumCircuit
        physical_qubits: list/tuple of physical qubit indices in the circuit
        gamma: QAOA parameter γ_p
        coeff: Hamiltonian coefficient (e.g. 1.0 or 2.0)
    """
    physical_qubits = list(physical_qubits)

    if len(physical_qubits) == 1:
        # 1-local: just an RZ on that qubit
        target = physical_qubits[0]
        qc.rz(2.0 * gamma * coeff, target)
        return

    target = physical_qubits[-1]
    controls = physical_qubits[:-1]

    # Compute parity of controls into target
    for q in controls:
        qc.cx(q, target)

    # RZ(2 * gamma * coeff) implements exp(-i * gamma * coeff * Z_target)
    # which, given the parity encoding, equals exp(-i * gamma * coeff * Z_prod)
    qc.rz(2.0 * gamma * coeff, target)

    # Uncompute parity
    for q in reversed(controls):
        qc.cx(q, target)


# ----------------------------------------------------------------------
# Helper 3: one LABS cost layer e^{-i gamma H_C} for arbitrary N
# ----------------------------------------------------------------------
def labs_cost_layer(qc: QuantumCircuit, gamma, labs_terms, logical_to_physical=None):
    """
    Apply one LABS phase separator layer e^{-i gamma H_C}.

    Args:
        qc: QuantumCircuit
        gamma: scalar or Parameter (e.g. gammas[p])
        labs_terms: list of (qubits, coeff) in logical indices 0..N-1
        logical_to_physical: list mapping logical index -> physical qubit in the circuit.
                             If None, assume logical index == physical index.
    """
    if logical_to_physical is None:
        logical_to_physical = list(range(qc.num_qubits))

    for logical_qubits, coeff in labs_terms:
        physical_qubits = [logical_to_physical[q] for q in logical_qubits]
        _apply_parity_phase(qc, physical_qubits, gamma, coeff)


# ----------------------------------------------------------------------
# Main: QAOA for LABS on IBM-style hardware
# ----------------------------------------------------------------------
def labs_qaoa_ibm(N, layers, max_nq, qubits_in_line=None):
    """
    Construct a QAOA circuit for the LABS problem of sequence length N.

    Args:
        N (int): LABS sequence length (number of logical qubits).
        layers (int): Number of QAOA layers (p).
        max_nq (int): Total number of physical qubits in the circuit.
                      Must satisfy max_nq >= N if using the first N in a line.
        qubits_in_line (list[int], optional):
            A list specifying the physical qubit order/topology (e.g. a linear chain).
            The first N entries will be used for the LABS instance.
            If None, we assume logical qubit i maps to physical qubit i.

    Returns:
        QuantumCircuit: QAOA circuit for LABS with N logical qubits.
    """
    if max_nq < N:
        raise ValueError(f"max_nq ({max_nq}) must be >= N ({N}).")

    if qubits_in_line is None:
        qubits_in_line = list(range(max_nq))

    # Logical qubits 0..N-1 mapped onto the first N entries of qubits_in_line
    logical_qubits = list(range(N))
    logical_to_physical = qubits_in_line[:N]

    # Parameter vectors
    betas = ParameterVector("betas", layers)
    gammas = ParameterVector("gammas", layers)

    # Precompute LABS Hamiltonian terms in logical indices
    labs_terms = build_labs_terms(N)

    # Build circuit: max_nq physical qubits, N classical bits
    qc = QuantumCircuit(max_nq, N)

    # Initial state: |+>^{⊗N} on the logical qubits
    qc.h(logical_to_physical)

    # QAOA layers
    for p in range(layers):
        # Cost layer: e^{-i gamma_p H_C}
        labs_cost_layer(qc, gammas[p], labs_terms, logical_to_physical)

        # Mixer layer: RX(-2 * beta_p) on each logical qubit
        for phys_q in logical_to_physical:
            qc.rx(-2.0 * betas[p], phys_q)

    # Measure logical qubits into classical bits 0..N-1
    qc.measure(logical_to_physical, list(range(N)))

    return qc


In [6]:
from get_objective import labs_true_optimal_energy
labs_true_optimal_energy(30)

LabsOptimumResult(n=30, energy=59, proven=True, status='KNOWN_PROVEN', sequence_pm1=None)

# Objective Function

In [7]:
# Compute LABS objective for a single bitstring.
# LABS objective: for spins s_i in {+1,-1}, E = sum_{k=1..N-1} ( sum_{i=0..N-k-1} s_i * s_{i+k} )^2
def cost_labs(bitstring: str) -> int:
    # map '1' -> +1, '0' -> -1 (matches other helpers in this codebase)
    spins = [1 if c == "1" else -1 for c in bitstring]
    N = len(spins)
    total = 0
    for k in range(1, N):
        A = sum(spins[i] * spins[i + k] for i in range(N - k))
        total += A * A
    return total

# Usage:
# energy = labs_energy_bitstring("1100")
# If you want to use the repo's objective_labs (which expects a samples dict),
# wrap the single string into a samples dict with count 1:
# repo_energy = objective_labs({ "1100": 1 }, results["optimal"][0])


def objective_labs(samples_dict, optimal):
    """
    Evaluates the performance of a quantum algorithm for the Max-Cut problem.

    Parameters:
    samples_dict (dict): A dictionary where keys are bitstrings (binary solutions), 
                         and values are their occurrence counts.
    G (networkx.Graph): The input weighted graph where edges represent cut costs.
    optimal (str): The optimal bitstring solution found by classical solvers (e.g., CPLEX).

    Returns:
    dict: A dictionary containing:
        - "results": A numpy array with computed cost, normalized cost ratio, and counts.
        - "min_cost": The cost of the optimal LABS solution.
        - "r": The expected approximation ratio.
        - "probability": The probability of sampling the optimal solution.
    """
    # Compute the cost of the optimal LABS solution
    min_cost = optimal

    results = []  # Stores results in the form [cost, ratio, counts]
    probability = 0  # Tracks probability of sampling the optimal solution

    # Iterate through all sampled bitstrings
    for bitstring, counts in samples_dict.items():
        cost = cost_labs(bitstring)  # Compute cost of the given bitstring
        r =  min_cost/cost  # Compute the cost ratio relative to the optimal solution
        results.append([cost, r, counts])  # Store results
        
        # If this bitstring matches the optimal cost, update probability
        if abs(cost - min_cost) < 1e-6:
            probability += counts
        
        # Check if a better-than-optimal solution appears (sanity check)
        if cost < min_cost:
            print(f"There is a better cost than that of the optimal solution: cost = {cost}, and min_cost = {min_cost}")

    # Convert results to a NumPy array for easy computation
    results = np.array(results)

    # Total number of shots (total sampled solutions)
    shots = np.sum(results[:, 2])

    # Compute the expected approximation ratio: (weighted sum of costs) / (shots * min_cost)
    rT = np.sum(results[:, 0] * results[:, 2]) / (shots * min_cost)

    # Normalize the probability of sampling the optimal solution
    probability /= shots

    # Return results in a structured dictionary
    return {
        "results": np.array(results),
        "min_cost": min_cost,
        "r": rT,
        "probability": probability
    }

def mitigate(samples_dict, G, random=False):
    """
    Applies error mitigation by flipping individual bits in sampled solutions 
    to find better Max-Cut solutions.

    Parameters:
    samples_dict (dict): A dictionary where keys are bitstrings (binary solutions), 
                         and values are their occurrence counts.
    G (networkx.Graph): The input weighted graph where edges represent cut costs.
    random (bool, optional): If True, randomizes the order in which qubits are flipped.
                             Default is False (systematic flipping).

    Returns:
    dict: A dictionary of improved bitstring samples with their updated counts.
    """

    # Define a mapping to flip bits ('0' -> '1', '1' -> '0')
    change = {"0": "1", "1": "0"}

    # Get the number of nodes (qubits)
    nq = G.number_of_nodes()

    # Extract weights from the graph's edges
    weights = {(i, j): (G[i][j]["weight"] if len(G[i][j]) != 0 else 1) for i, j in G.edges}

    # Dictionary to store new (improved) samples
    new_samples = defaultdict(int)

    # Iterate over all bitstring samples
    for bitstring, counts in samples_dict.items():
        for _ in range(counts):  # Process each occurrence of the bitstring separately
            best_string = bitstring  # Initialize the best solution as the current one
            best_cost = cost_maxcut(bitstring, weights)  # Compute its cost
            
            # Create an ordered list of qubits (nodes) to consider flipping
            list_qubits = np.arange(nq)
            
            # If random flipping is enabled, shuffle the qubit order
            if random:
                np.random.shuffle(list_qubits)

            # Try flipping each qubit and check if the cost improves
            for qi in list_qubits:
                # Flip the bit at position qi
                new_string = "".join((change[i] if n == qi else i) for n, i in enumerate(best_string))
                new_cost = cost_maxcut(new_string, weights)

                # If the new configuration gives a better cost, update the best solution
                if new_cost > best_cost:
                    best_string = new_string
                    best_cost = new_cost
            
            # Store the improved bitstring in the new_samples dictionary
            new_samples[best_string] += 1

    return new_samples  # Return the mitigated samples

def random_samples(num_samples, n_qubits):
    """
    Generates random bitstring samples for a given number of qubits.

    Parameters:
    num_samples (int): The number of random bitstrings to generate.
    n_qubits (int): The number of qubits (length of each bitstring).

    Returns:
    dict: A dictionary where keys are randomly generated bitstrings 
          and values are their occurrence counts.
    """
    
    random_samples = defaultdict(int)  # Dictionary to store bitstrings and their counts

    # Generate random bitstrings and count their occurrences
    for _ in range(num_samples):
        bitstring = "".join(str(i) for i in np.random.choice([0, 1], n_qubits))  # Generate a random bitstring
        random_samples[bitstring] += 1  # Increment count for the generated bitstring

    return random_samples  # Return the dictionary of samples


def mitigate_labs(samples_dict, nq, random=False):
    """
    Error mitigation for LABS: flips individual bits to (locally) improve LABS objective.

    Parameters:
    samples_dict (dict): keys are bitstrings, values are counts.
    nq (int): number of qubits (length of bitstrings).
    random (bool): if True, randomize flip order.

    Returns:
    dict: improved samples dict (bitstring -> counts)
    """
    change = {"0": "1", "1": "0"}
    new_samples = defaultdict(int)

    for bitstring, counts in samples_dict.items():
        for _ in range(counts):
            best_string = bitstring
            best_cost = cost_labs(bitstring)  # LABS objective: lower is better

            list_qubits = np.arange(nq)
            if random:
                np.random.shuffle(list_qubits)

            for q in list_qubits:
                s = list(best_string)
                s[q] = change[s[q]]
                cand = "".join(s)
                cand_cost = cost_labs(cand)
                # For LABS we minimize the objective
                if cand_cost < best_cost:
                    best_cost = cand_cost
                    best_string = cand

            new_samples[best_string] += 1

    return new_samples

def mitigate(samples_dict, G, random=False):
    """
    Applies error mitigation by flipping individual bits in sampled solutions 
    to find better Max-Cut solutions.

    Parameters:
    samples_dict (dict): A dictionary where keys are bitstrings (binary solutions), 
                         and values are their occurrence counts.
    G (networkx.Graph): The input weighted graph where edges represent cut costs.
    random (bool, optional): If True, randomizes the order in which qubits are flipped.
                             Default is False (systematic flipping).

    Returns:
    dict: A dictionary of improved bitstring samples with their updated counts.
    """

    # Define a mapping to flip bits ('0' -> '1', '1' -> '0')
    change = {"0": "1", "1": "0"}

    # Get the number of nodes (qubits)
    nq = G.number_of_nodes()

    # Extract weights from the graph's edges
    weights = {(i, j): (G[i][j]["weight"] if len(G[i][j]) != 0 else 1) for i, j in G.edges}

    # Dictionary to store new (improved) samples
    new_samples = defaultdict(int)

    # Iterate over all bitstring samples
    for bitstring, counts in samples_dict.items():
        for _ in range(counts):  # Process each occurrence of the bitstring separately
            best_string = bitstring  # Initialize the best solution as the current one
            best_cost = cost_maxcut(bitstring, weights)  # Compute its cost
            
            # Create an ordered list of qubits (nodes) to consider flipping
            list_qubits = np.arange(nq)
            
            # If random flipping is enabled, shuffle the qubit order
            if random:
                np.random.shuffle(list_qubits)

            # Try flipping each qubit and check if the cost improves
            for qi in list_qubits:
                # Flip the bit at position qi
                new_string = "".join((change[i] if n == qi else i) for n, i in enumerate(best_string))
                new_cost = cost_maxcut(new_string, weights)

                # If the new configuration gives a better cost, update the best solution
                if new_cost > best_cost:
                    best_string = new_string
                    best_cost = new_cost
            
            # Store the improved bitstring in the new_samples dictionary
            new_samples[best_string] += 1

    return new_samples  # Return the mitigated samples

def random_samples(num_samples, n_qubits):
    """
    Generates random bitstring samples for a given number of qubits.

    Parameters:
    num_samples (int): The number of random bitstrings to generate.
    n_qubits (int): The number of qubits (length of each bitstring).

    Returns:
    dict: A dictionary where keys are randomly generated bitstrings 
          and values are their occurrence counts.
    """
    
    random_samples = defaultdict(int)  # Dictionary to store bitstrings and their counts

    # Generate random bitstrings and count their occurrences
    for _ in range(num_samples):
        bitstring = "".join(str(i) for i in np.random.choice([0, 1], n_qubits))  # Generate a random bitstring
        random_samples[bitstring] += 1  # Increment count for the generated bitstring

    return random_samples  # Return the dictionary of samples


In [8]:
problems = np.load("./Data/problems_1DChain_LABS.npy", allow_pickle=True).item()

# 1) IBM Eagle and Heron: Prepare the experiments - LABS

In [9]:
##jm
#labs_flag = True
#labs_sizes = [5,6]  # LABS problem sizes; single value for easy extension
#N = labs_sizes[0]
## Select the backend for quantum computation
## Uncomment the desired backend or set the active one
## backend_name = "ibm_brisbane"
## backend_name = "ibm_sherbrooke"
## backend_name = "ibm_kyiv"
## backend_name = "ibm_nazca"
## backend_name = "ibm_osaka"
## backend_name = "ibm_kyoto"
## backend_name = "ibm_torino"
## backend_name = "ibm_fez"  # Active backend
## backend_name = "ibm_marrakesh"
## backend_name = "ibm_strasbourg"
## backend_name = "ibm_brussels"
## backend_name = "ankaa_2"
#backend_name = "qasm_simulator"  # Classical simulator option
#
#results = {}  # Dictionary to store experiment results
#
## ---------------------- Loading Problem Information ----------------------
## jm
#nq = N  # Number of qubits (size of the problem); matches LABS size for now
#
## Define qubit mappings for different backends
##qubits_in_line["qasm_simulator"] = range(nq)  # Simulator uses a linear qubit range
#
## Load the problem graph
##results["G"] = problems["G"][nq]  
#
## Repeat subgraph structure to match the qubit layout of the chosen backend
##results["GT"] = repeat_subgraph(results["G"], qubits_in_line[backend_name])
#
## Select the graph to use: 
## If using a real quantum backend, use the transformed graph (GT), 
## otherwise use the original problem graph (G) for the simulator
##GT = results["GT"] if backend_name != "qasm_simulator" else results["G"]
##print('using graph with ', GT.number_of_nodes(), ' nodes and ', GT.number_of_edges(), ' edges')
## Load optimal solutions for the given problem size
#sols = problems["sol"][nq]  
#
## Get the list of qubits assigned for the selected backend
## qubits_line = qubits_in_line[backend_name]
#
## Get the maximum number of qubits available on the backend
#max_nq = backends[backend_name].num_qubits  
#
## ---------------------- QAOA Configuration ----------------------
#
## Define the number of QAOA layers
##ps = [3, 4, 5, 6, 7, 8, 9, 10, 13, 15, 20, 25, 30, 40, 50, 75, 100]  
##jm
#ps = [3, 4, 5] #, 4, 5, 6, 7, 8, 9, 10, 13, 15, 20, 25, 30, 40, 50, 75, 100]  
## ps = [10_000]  # Set a single large QAOA depth for testing
#
## Define delta values for parameter scaling
## deltas = np.linspace(0.5, 1.5, 10)  # Generate 10 values between 0.5 and 1.5
#deltas = [0.75]  # Set a single delta value for testing
#
## Store key experiment parameters in results
#results["optimal"] = sols  
#results["nq"] = nq  
#results["ps"] = ps  
#results["Deltas"] = deltas  
#
## ---------------------- Circuit Compilation ----------------------
#
#circuits_transpiled = {}  # Store transpiled circuits for different QAOA layers
#
## Iterate over different QAOA layers
#for p in ps:
#    print(f"Layer: -------  {p} ----------- ")  # Display current layer
#    if labs_flag:
#      layers=p
#      circ_maxcut = labs_qaoa_ibm(N, layers, max_nq, qubits_in_line=None)
#    else: 
#      # Generate the QAOA circuit for MaxCut problem
#      circ_maxcut = maxcut_qaoa_ibm(GT, p, max_nq, qubits_line, fractional_gates=True)
#    
#    # Transpile the circuit for the selected backend
#    # Optimization level 1 applies basic circuit optimizations while preserving logical structure
#    backend_circ = transpile(circ_maxcut, backend=backends[backend_name], optimization_level=1, initial_layout=range(max_nq))
#    
#    # Store the transpiled circuit
#    circuits_transpiled[p] = backend_circ  
#
## ---------------------- Assigning Parameters & Executing Circuits ----------------------
#
#circuits = []  # Store final circuits with assigned parameters
#
## Iterate over different delta values
#for delta in deltas:
#    print(f"Delta: -------  {round(delta,2)} ----------- ")  # Display current delta value
#    
#    for p in ps:
#        # Compute beta and gamma parameters for QAOA
#        betas = list(np.arange(1, p+1)[::-1] * delta/p)  # Reverse sequence for betas
#        gammas = list(np.arange(1, p+1) * delta/p)  # Forward sequence for gammas
#        
#        # Assign the computed parameters to the transpiled circuit
#        backend_circ = circuits_transpiled[p].assign_parameters(np.concatenate((betas, gammas)))
#        
#        # Store the final circuit
#        circuits.append(backend_circ)  
#

In [38]:
# jm
labs_flag = True
labs_sizes = [5, 6]  # LABS problem sizes

# backend
backend_name = "qasm_simulator"  # Classical simulator option
max_nq = backends[backend_name].num_qubits

# QAOA config
ps = [3, 4, 5]
deltas = [0.75]

all_results = {}
circuits_by_nq = {}

for N in labs_sizes:
    nq = N
    sols = problems["sol"][nq]

    results = {}
    results["optimal"] = sols
    results["nq"] = nq
    results["ps"] = ps
    results["Deltas"] = deltas

    # Compile circuits per p
    circuits_transpiled = {}
    for p in ps:
        print(f"N={N} | Layer: -------  {p} ----------- ")
        if labs_flag:
            circ_maxcut = labs_qaoa_ibm(N, p, max_nq, qubits_in_line=None)
        else:
            circ_maxcut = maxcut_qaoa_ibm(GT, p, max_nq, qubits_line, fractional_gates=True)

        backend_circ = transpile(
            circ_maxcut,
            backend=backends[backend_name],
            optimization_level=1,
            initial_layout=range(max_nq),
        )
        circuits_transpiled[p] = backend_circ

    # Assign parameters per delta
    circuits = []
    for delta in deltas:
        print(f"N={N} | Delta: -------  {round(delta,2)} ----------- ")
        for p in ps:
            betas = list(np.arange(1, p + 1)[::-1] * delta / p)
            gammas = list(np.arange(1, p + 1) * delta / p)
            backend_circ = circuits_transpiled[p].assign_parameters(
                np.concatenate((betas, gammas))
            )
            circuits.append(backend_circ)

    circuits_by_nq[nq] = circuits
    all_results[nq] = results


N=5 | Layer: -------  3 ----------- 
N=5 | Layer: -------  4 ----------- 
N=5 | Layer: -------  5 ----------- 
N=5 | Delta: -------  0.75 ----------- 
N=6 | Layer: -------  3 ----------- 
N=6 | Layer: -------  4 ----------- 
N=6 | Layer: -------  5 ----------- 
N=6 | Delta: -------  0.75 ----------- 


In [39]:
circuits_by_nq

{5: [<qiskit.circuit.quantumcircuit.QuantumCircuit at 0x14948e150>,
 6: [<qiskit.circuit.quantumcircuit.QuantumCircuit at 0x106deafc0>,
  <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x149060cb0>]}

In [40]:
all_results

{5: {'optimal': 2, 'nq': 5, 'ps': [3, 4, 5], 'Deltas': [0.75]},
 6: {'optimal': 7, 'nq': 6, 'ps': [3, 4, 5], 'Deltas': [0.75]}}

# 2) IBM Eagle and Heron: Run the experiments

In [41]:
circuits

In [42]:
# shots = 1_000
shots = 1
extra = ""

for nq in labs_sizes:
    results = all_results[nq]
    results["shots"] = shots
    circuits = circuits_by_nq[nq]

    if backend_name != "qasm_simulator":
        sampler = Sampler(mode=backends[backend_name])
        submit_job = sampler.run(circuits, shots=shots)
        results["id"] = submit_job.job_id()
    else:
        print("running with backend_name:", backend_name)
        result = backends[backend_name].run(circuits, shots=shots).result()
        dict_results = [result.get_counts(i) for i in range(len(circuits))]

        results["samples"] = {
            delta: {
                p: {k[::-1]: v for k, v in dict_results[i + nd * len(ps)].items()}
                for i, p in enumerate(results["ps"])
            }
            for nd, delta in enumerate(results["Deltas"])
        }

    np.save(f"./Data/{backend_name}/{nq}_1D{extra}_LABS.npy", results)
    print("results saved to", f"./Data/{backend_name}/{nq}_1D{extra}_LABS.npy")


running with backend_name: qasm_simulator
results saved to ./Data/qasm_simulator/5_1D_LABS.npy
running with backend_name: qasm_simulator
results saved to ./Data/qasm_simulator/6_1D_LABS.npy


In [43]:
results

{'optimal': 7,
 'nq': 6,
 'ps': [3, 4, 5],
 'Deltas': [0.75],
 'shots': 1,
 'samples': {0.75: {3: {'101101': 1}, 4: {'011111': 1}, 5: {'111110': 1}}}}

In [44]:
results['samples']

{0.75: {3: {'101101': 1}, 4: {'011111': 1}, 5: {'111110': 1}}}

In [45]:
results['ps']

[3, 4, 5]

# Retrieve the experimental information

In [46]:
#backend_name = "ibm_fez"  # Define the IBM backend used for execution
#jm
backend_name='qasm_simulator'

# Specify additional identifier for file naming:
extra = ""  # No extra identifier

all_results = {}

for nq in labs_sizes:
    # Load the previously saved results from a NumPy binary file
    results = np.load(f"./Data/{backend_name}/{nq}_1D{extra}_LABS.npy", allow_pickle=True).item()

    # If using a real quantum device, retrieve job results from IBM Quantum service
    if backend_name != "qasm_simulator":
        # Fetch the job results using its stored job ID
        job = service.job(job_id=results["id"]).result()

        # Extract bitstring measurement results from all circuits in the job
        dict_results = [job[i].data.c.get_counts() for i in range(len(job))]

        # Process and store results:
        # - Reverse bitstrings (`k[::-1]`) to match standard qubit ordering.
        # - Organize results into a nested dictionary: {delta -> {p -> {bitstring -> counts}}}.
        results["samples"] = {
            delta: {
                p: {k[::-1]: v for k, v in dict_results[i + nd * len(ps)].items()}
                for i, p in enumerate(results["ps"])
            }
            for nd, delta in enumerate(results["Deltas"])
        }

    all_results[nq] = results

# Keep the last nq results in `results` for downstream cells
results = all_results[nq]


In [47]:
results

{'optimal': 7,
 'nq': 6,
 'ps': [3, 4, 5],
 'Deltas': [0.75],
 'shots': 1,
 'samples': {0.75: {3: {'101101': 1}, 4: {'011111': 1}, 5: {'111110': 1}}}}

# Postprocessing the samples - LABS

In [48]:
results['optimal']

7

In [49]:
# Extract the number of nodes (qubits) in the original and repeated graph
#nq = results["G"].number_of_nodes()  # Number of qubits in the original problem graph
#nq_total = results["GT"].number_of_nodes()  # Number of qubits in the repeated subgraph
#sections = nq_total // nq  # Number of repeated sections in the larger subgraph

# Dictionaries to store post-processing results
postprocessing = {}
postprocessing_mitig = {}

# Iterate over different values of delta (hyperparameter for QAOA)
for delta in results["samples"]:
    postprocessing[delta] = {}
    postprocessing_mitig[delta] = {}

    # Iterate over different QAOA layer depths (p)
    for p in results["samples"][delta]:
        print(f"----------- p = {p} -------------")
        postprocessing[delta][p] = {}
        postprocessing_mitig[delta][p] = {}

        samples_sec = results["samples"][delta][p]

        # Compute objective function for the labs problem
        if labs_flag:
            postprocessing[delta][p] = objective_labs(samples_sec, results["optimal"])
            # Apply error mitigation (local search improvement)
            new_samples = mitigate_labs(samples_sec, nq, random=False)
            postprocessing_mitig[delta][p] = objective_labs(new_samples, results["optimal"])

# Store the post-processed and mitigated results
results["postprocessing"] = postprocessing
results["postprocessing_mitig"] = postprocessing_mitig

# Generate random bitstring samples for comparison
#rand_samples = random_samples(10_000, nq)
rand_samples = random_samples(10, nq)

# Compute the objective function for random sampling (baseline performance)
results["random"] = objective_labs(rand_samples, results["optimal"])
results["random_mitig"] = objective_labs(mitigate_labs(rand_samples, nq, random=False), results["optimal"])

# Save the updated results dictionary
np.save(f"./Data/{backend_name}/{nq}_1D{extra}_LABS.npy", results)

print(results)


----------- p = 3 -------------
----------- p = 4 -------------
----------- p = 5 -------------
{'optimal': 7, 'nq': 6, 'ps': [3, 4, 5], 'Deltas': [0.75], 'shots': 1, 'samples': {0.75: {3: {'101101': 1}, 4: {'011111': 1}, 5: {'111110': 1}}}, 'postprocessing': {0.75: {3: {'results': array([[23.        ,  0.30434783,  1.        ]]), 'min_cost': 7, 'r': np.float64(3.2857142857142856), 'probability': np.float64(0.0)}, 4: {'results': array([[15.        ,  0.46666667,  1.        ]]), 'min_cost': 7, 'r': np.float64(2.142857142857143), 'probability': np.float64(0.0)}, 5: {'results': array([[15.        ,  0.46666667,  1.        ]]), 'min_cost': 7, 'r': np.float64(2.142857142857143), 'probability': np.float64(0.0)}}}, 'postprocessing_mitig': {0.75: {3: {'results': array([[7., 1., 1.]]), 'min_cost': 7, 'r': np.float64(1.0), 'probability': np.float64(1.0)}, 4: {'results': array([[7., 1., 1.]]), 'min_cost': 7, 'r': np.float64(1.0), 'probability': np.float64(1.0)}, 5: {'results': array([[7., 1., 1.]

In [32]:
# bitstring results from the experiments
results['samples']

{0.75: {3: {'100010': 1}, 4: {'001101': 1}, 5: {'000011': 1}}}

In [33]:
# random bitstring results
results['postprocessing']

{0.75: {3: {'results': array([[7., 1., 1.]]),
   'min_cost': 7,
   'r': np.float64(1.0),
   'probability': np.float64(1.0)},
  4: {'results': array([[7., 1., 1.]]),
   'min_cost': 7,
   'r': np.float64(1.0),
   'probability': np.float64(1.0)},
  5: {'results': array([[15.        ,  0.46666667,  1.        ]]),
   'min_cost': 7,
   'r': np.float64(2.142857142857143),
   'probability': np.float64(0.0)}}}

In [37]:
from pprint import pprint

# Example: get results for delta=0.75, p=5
delta = 0.75
p = 5

print("raw results:")
pprint(results["samples"][delta][p], width=120)

print("\npostprocessing results:")
pprint(results["postprocessing"][delta][p], width=120)


raw results:
{'000011': 1}

postprocessing results:
{'min_cost': 7,
 'probability': np.float64(0.0),
 'r': np.float64(2.142857142857143),
 'results': array([[15.        ,  0.46666667,  1.        ]])}
